# 05pagedata: data for the interactive page

**What it does.** Turns the clean tables into three small JSON files that the interactive page (`docs/index.html`) reads, and checks each one against the clean tables.

- `stations.json`: for each KTX or SRT station, monthly KTX riders (2004 to 2022) and yearly KTX and SRT riders (2016 to 2024), for the station explorer.
- `timeline.json`: yearly and monthly high-speed riders with the main events, for the event timeline.
- `metro_day.json`: the share of each hour in the day's boardings and alightings, for the network and each metro rail line, for the metro day view.

**Inputs:** the clean tables in `data/clean` and `data/names_en.csv`.
**Outputs:** `docs/data/stations.json`, `docs/data/timeline.json`, `docs/data/metro_day.json`.

**Data credit:** Korea Railroad Corporation (KORAIL), via data.go.kr, railstat.korail.com and info.korail.com.

## 0. Settings

In [1]:
from pathlib import Path
import json
import pandas as pd

CLEAN = Path("data/clean")  # <- clean tables
NAMES = Path("data/names_en.csv")  # <- English names
OUT = Path("docs/data")  # <- folder the page reads from

METRO_YEAR = 2025  # <- year shown in the metro day view
MIN_LINE_BOARDINGS_M = 4  # <- metro lines with fewer boardings a year (millions) are left out of the line list

EVENTS = []  # events shown on the timelines
EVENTS.append({"date": "2004-04", "label": "KTX opens (Gyeongbu, Honam)"})  # <-
EVENTS.append({"date": "2010-11", "label": "Gyeongbu HSR to Busan completed"})  # <-
EVENTS.append({"date": "2015-04", "label": "Honam HSR opens"})  # <-
EVENTS.append({"date": "2016-12", "label": "SRT starts from Suseo"})  # <-
EVENTS.append({"date": "2017-12", "label": "Gangneung line opens"})  # <-
EVENTS.append({"date": "2020-02", "label": "COVID-19"})  # <-
EVENTS.append({"date": "2022-04", "label": "Distancing rules end"})  # <-

OUT.mkdir(parents=True, exist_ok=True)
names = pd.read_csv(NAMES)
EN = dict(zip(names["ko"], names["en"]))
print("events:", len(EVENTS))

events: 7


In [2]:
def save(obj, name):
    path = OUT / name
    text = json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False)  # browsers reject NaN, so stop if any is left
    path.write_text(text, encoding="utf-8")
    print(name, round(path.stat().st_size / 1024, 1), "KB")


def r1(x):
    return None if pd.isna(x) else round(float(x), 1)

## 1. Stations

Monthly values are KTX boardings plus alightings in thousands (data.go.kr file; 50 stations, today's names; stations under 500 a month show as 0). Yearly values are boardings plus alightings in thousands from the statistics portal, KTX and SRT separately. Seoul's yearly KTX figures include Gangneung line trains, the monthly ones do not.

In [3]:
ktx_month = pd.read_csv(CLEAN / "ktx_station_month.csv", parse_dates=["month"])
wide = ktx_month.pivot_table(index="month", columns="station", values="riders_k")
months = [m.strftime("%Y-%m") for m in wide.index]

yearly = pd.read_csv(CLEAN / "station_year_ktx_srt.csv")
yearly["station"] = yearly["station"].replace({"여수엑수포": "여수엑스포"})
yearly["riders_k"] = (yearly["board"].fillna(0) + yearly["alight"].fillna(0)) / 1000
yearly = yearly.groupby(["operator", "station", "year"])["riders_k"].sum().unstack("year")
years = [int(y) for y in yearly.columns]

all_names = sorted(set(wide.columns) | set(yearly.loc["KTX"].index) | set(yearly.loc["SRT"].index))
missing = [s for s in all_names if s not in EN]
print("stations:", len(all_names), "| without an English name:", missing)

stations: 88 | without an English name: []


In [4]:
station_list = []
for s in all_names:
    item = {"ko": s, "en": EN.get(s, s)}
    if s in wide.columns:
        item["monthly"] = [r1(v) for v in wide[s]]
    if s in yearly.loc["KTX"].index:
        item["ktx"] = [r1(v) for v in yearly.loc[("KTX", s)]]
    if s in yearly.loc["SRT"].index:
        item["srt"] = [r1(v) for v in yearly.loc[("SRT", s)]]
    latest = 0
    if "ktx" in item:
        latest += item["ktx"][-1] or 0
    if "srt" in item:
        latest += item["srt"][-1] or 0
    item["latest_k"] = round(latest, 1)
    station_list.append(item)

station_list = sorted(station_list, key=lambda d: -d["latest_k"])
stations_json = {"months": months, "years": years, "unit": "boardings plus alightings, thousands", "stations": station_list}
save(stations_json, "stations.json")
print("busiest five in", years[-1], ":", [(d["ko"], d["latest_k"]) for d in station_list[:5]])

stations.json 67.7 KB
busiest five in 2024 : [('서울', 37777.9), ('부산', 20358.6), ('동대구', 18504.1), ('대전', 16275.5), ('수서', 16067.1)]


In [5]:
busan = next(d for d in station_list if d["ko"] == "부산")
check_ktx = yearly.loc[("KTX", "부산"), 2019]
check_month = wide.loc["2019", "부산"].sum()
print("Busan KTX 2019, json:", busan["ktx"][years.index(2019)], "| clean table:", round(check_ktx, 1))
print("Busan monthly 2019 sum, json:", round(sum(v for v, m in zip(busan["monthly"], months) if m.startswith("2019")), 1), "| clean table:", round(check_month, 1))

Busan KTX 2019, json: 11974.0 | clean table: 11974.0
Busan monthly 2019 sum, json: 11973.0 | clean table: 11973.0


## 2. Timeline

Yearly high-speed boardings (millions): the line table for 2004 to 2015, the station table from 2016, KTX and SRT separately. Monthly KTX and SRT boardings (millions) for 2016 to 2023 from the monthly table; SRT is 0 before December 2016. JSON cannot hold NaN (browsers reject it), so `save` stops if any is left.

In [6]:
line_year = pd.read_csv(CLEAN / "line_year.csv")
early = line_year[(line_year["line"] == "합계") & (line_year["type"] == "KTX") & (line_year["year"] < 2016)].set_index("year")["riders"]
st = pd.read_csv(CLEAN / "station_year_ktx_srt.csv")
late = st.groupby(["year", "operator"])["board"].sum().unstack("operator")
hs_year = pd.DataFrame({"KTX": pd.concat([early, late["KTX"]]), "SRT": late["SRT"]}).fillna(0) / 1e6

riders_m = pd.read_csv(CLEAN / "riders_month.csv", parse_dates=["month"], index_col="month")
riders_m = riders_m.fillna(0)  # SRT has no months before December 2016

yearly_part = {"years": [int(y) for y in hs_year.index], "ktx": [round(v, 2) for v in hs_year["KTX"]], "srt": [round(v, 2) for v in hs_year["SRT"]]}
monthly_part = {"months": [m.strftime("%Y-%m") for m in riders_m.index], "ktx": [round(v / 1e6, 3) for v in riders_m["KTX"]], "srt": [round(v / 1e6, 3) for v in riders_m["SRT"]]}
timeline_json = {"yearly": yearly_part, "monthly": monthly_part, "unit": "boardings, millions", "events": EVENTS}
save(timeline_json, "timeline.json")
print("2019:", round(hs_year.loc[2019].sum(), 1), "m | 2024:", round(hs_year.loc[2024].sum(), 1), "m")
print("monthly sum 2023 against yearly 2023:", round(sum(timeline_json["monthly"]["ktx"][-12:]) + sum(timeline_json["monthly"]["srt"][-12:]), 1), "|", round(hs_year.loc[2023].sum(), 1))

timeline.json 2.8 KB
2019: 94.2 m | 2024: 116.6 m
monthly sum 2023 against yearly 2023: 110.4 | 110.4


## 3. Metro day

Gate taps by hour, all months of the chosen year together, as a percent of the day. Hour 6 means before 07:00, hour 24 after midnight. Lines with very few riders are left out of the line list but stay in the network total.

In [7]:
hourly = pd.read_csv(CLEAN / "metro_hourly.csv", parse_dates=["month"])
h = hourly[hourly["month"].dt.year == METRO_YEAR]
hours = sorted(h["hour"].unique().tolist())


def profile(df):
    p = df.groupby("hour")[["board", "alight"]].sum()
    total = p.sum()
    p = p / total * 100
    return {"board": [round(v, 2) for v in p["board"]], "alight": [round(v, 2) for v in p["alight"]], "boardings_m": round(total["board"] / 1e6, 1)}


lines = []
for line, df in h.groupby("line"):
    prof = profile(df)
    if prof["boardings_m"] < MIN_LINE_BOARDINGS_M:
        continue
    prof["ko"] = line
    prof["en"] = EN.get(line, line)
    prof["morning_peak"] = round(prof["board"][hours.index(7)] + prof["board"][hours.index(8)], 1)
    lines.append(prof)
lines = sorted(lines, key=lambda d: -d["morning_peak"])

network = profile(h)
network["ko"] = "광역철도 전체"
network["en"] = "All metro rail"
network["morning_peak"] = round(network["board"][hours.index(7)] + network["board"][hours.index(8)], 1)
metro_json = {"year": METRO_YEAR, "hours": hours, "unit": "percent of the day's gate taps", "network": network, "lines": lines}
save(metro_json, "metro_day.json")
print("lines:", [(d["en"], d["morning_peak"]) for d in lines])
print("network shares add to:", round(sum(network["board"]), 1), "and", round(sum(network["alight"]), 1))

metro_day.json 4.9 KB
lines: [('Jungang Line', 24.5), ('Gyeonggang Line', 23.8), ('Gyeongin Line', 23.0), ('Gyeongui Line', 22.7), ('Seohae Line', 22.5), ('Gyeongchun Line', 21.9), ('Ilsan Line', 21.7), ('Ansan Line', 20.9), ('Suin Line', 20.4), ('Gyeongwon Line', 19.5), ('Gwacheon Line', 18.5), ('Bundang Line', 15.8), ('Gyeongbu Line', 15.7), ('Donghae Line', 15.6), ('Janghang Line', 13.9), ('Daegyeong Line', 11.2)]
network shares add to: 100.0 and 100.0


## 4. Files

In [8]:
for f in sorted(OUT.glob("*.json")):
    data = json.loads(f.read_text(encoding="utf-8"))
    print(f.name, round(f.stat().st_size / 1024, 1), "KB | keys:", list(data.keys()))

metro_day.json 4.9 KB | keys: ['year', 'hours', 'unit', 'network', 'lines']
stations.json 67.7 KB | keys: ['months', 'years', 'unit', 'stations']
timeline.json 2.8 KB | keys: ['yearly', 'monthly', 'unit', 'events']


___
**END**